In [0]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, current_date, lit
from pyspark.sql.utils import AnalysisException

#Definição dos caminhos dos arquivos
path_meta_brasil = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv"
path_meta_municipio = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv"
path_meta_uf = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv"
path_avaliacao_municipio = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_municipio.csv"
path_avaliacao_uf = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_uf.csv"
path_avaliacao_alunos = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_alunos.csv"

### METAS BRASIL - IMPORTAÇÃO E ANÁLISE _EXPLORATÓRIA_

In [0]:
df_spark = (spark.read
            .format("csv")
            .option("header", "true")      # Usa a primeira linha como cabeçalho
            .option("inferSchema", "true") # Identifica automaticamente os tipos de dados (int, string, etc.)
            .option("sep", ",")            # Altere para ";" se o seu CSV usar ponto e vírgula
            .load(path_meta_brasil))
display(df_spark)

#Como só tem 3 linhas, não tem muito o que fazer aqui

### METAS UF - IMPORTAÇÃO E ANÁLISE EXPLORATÓRIA

In [0]:

df_spark = (spark.read
            .format("csv")
            .option("header", "true")      # Usa a primeira linha como cabeçalho
            .option("inferSchema", "true") # Identifica automaticamente os tipos de dados (int, string, etc.)
            .option("sep", ",")            # Altere para ";" se o seu CSV usar ponto e vírgula
            .load(path_meta_uf))
display(df_spark)

Só com o display já é possível notar que Roraima não tem informações de taxa de alfabetização e nem metas.

O Acre e DF não possuem algumas informações (valores null).

###METAS MUNICÍPIO - IMPORTAÇÃO E ANÁLISE EXPLORATÓRIA

In [0]:
df_spark = (spark.read
            .format("csv")
            .option("header", "true")      # Usa a primeira linha como cabeçalho
            .option("inferSchema", "true") # Identifica automaticamente os tipos de dados (int, string, etc.)
            .option("sep", ",")            # Altere para ";" se o seu CSV usar ponto e vírgula
            .load(path_meta_municipio))
display(df_spark)

O primeiro ponto que me incomoda é que esta tabela não tem a UF dos municípios. Alguns id_municipio pelo display não possuem informações das metas e taxa - mas não sei se são cidades de RORAIMA, já que este estado não possui estas informações. Vou verificar nos outros arquivos se em algum dele possui este de-para, pois se não terei que importar mais uma tabela externa para o projeto.


### AVALIAÇÃO UF - IMPORTAÇÃO E ANÁLISE EXPLORATÓRIA

In [0]:
df_spark = (spark.read
            .format("csv")
            .option("header", "true")      # Usa a primeira linha como cabeçalho
            .option("inferSchema", "true") # Identifica automaticamente os tipos de dados (int, string, etc.)
            .option("sep", ",")            # Altere para ";" se o seu CSV usar ponto e vírgula
            .load(path_avaliacao_uf))
display(df_spark)

In [0]:
from pyspark.sql import functions as F
df_nulos = (df_spark.groupBy("ano", "sigla_uf")
            .agg(
                F.sum(F.col("taxa_alfabetizacao").isNull().cast("int")).alias("nulos_taxa_alfabetizacao"),
                F.sum(F.col("media_portugues").isNull().cast("int")).alias("nulos_media_portugues"),
                F.sum(F.col("proporcao_aluno_nivel_0").isNull().cast("int")).alias("nulos_proporcao_aluno_nivel_0")
            ))
display(df_nulos)

Olhado o agrupamento realizado, RR não está presente neste dataframe e, assim como observado na tabela de metas por UF, o ACRE só tem informações de 2024. 

DF não está presente aqui nesta tabela

Também é possível notar que há uma quantidade grande de nulos nas colunas de proporção alunos por nível. Abaixo vou fazer de forma massiva por coluna a quantidade de nulls por UF.

In [0]:
#Tentei fazer a análise "na unha", mas aí perguntei pro GPT se havia uma forma de verificar as colunas agrupadas de forma massiva os 'nulls'
#Avaliador tenha piedade de mim.

# 1. Identifica automaticamente todas as colunas de notas (remove as colunas de agrupamento)
colunas_agrupamento = ["sigla_uf"]
colunas_notas = [c for c in df_spark.columns if c not in colunas_agrupamento]

# 2. Cria dinamicamente a lista de agregações para cada coluna
expressoes_agg = [
    F.sum(F.col(c).isNull().cast("int")).alias(f"nulos_{c}") 
    for c in colunas_notas
]

# 3. Executa o agrupamento passando a lista gerada (*expressoes_agg)
df_nulos = df_spark.groupBy(colunas_agrupamento).agg(*expressoes_agg)

# Exibe o resultado na tela
display(df_nulos)

Para esta tabela, as únicas colunas que possuem os dados preenchidos são a série, rede, taxa de alfabetização e média português.

### AVALIAÇÃO MUNICÍPIO - IMPORTAÇÃO E ANÁLISE EXPLORATÓRIA

In [0]:
df_spark = (spark.read
            .format("csv")
            .option("header", "true")      # Usa a primeira linha como cabeçalho
            .option("inferSchema", "true") # Identifica automaticamente os tipos de dados (int, string, etc.)
            .option("sep", ",")            # Altere para ";" se o seu CSV usar ponto e vírgula
            .load(path_avaliacao_municipio))
display(df_spark)

In [0]:
# Cria uma contagem condicional para cada coluna do DataFrame
df_resumo_nulos = df_spark.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_spark.columns
])

# Exibe o resultado em formato de tabela no Databricks
display(df_resumo_nulos)


Mesmo comportamento da tabela de UF, com muitos valores nulos nas colunas de proporção_aluno_nivel...

E SEM O DE-PARA DE UF!!!! PELO JEITO TEREMOS QUE IR ATRÁS DESSA BASE PARA PODERMOS CRUZAR AS INFORMAÇÕES!!!

### AVALIAÇÃO ALUNOS - IMPORTAÇÃO E ANÁLISE EXPLORATÓRIA

In [0]:
df_spark = (spark.read
            .format("csv")
            .option("header", "true")      # Usa a primeira linha como cabeçalho
            .option("inferSchema", "true") # Identifica automaticamente os tipos de dados (int, string, etc.)
            .option("sep", ",")            # Altere para ";" se o seu CSV usar ponto e vírgula
            .load(path_avaliacao_alunos))
display(df_spark)

In [0]:
total_linhas = df_spark.count()
total_colunas = len(df_spark.columns)

print(f"Linhas: {total_linhas} | Colunas: {total_colunas}")

In [0]:
# Cria uma contagem condicional para cada coluna do DataFrame
df_resumo_nulos = df_spark.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_spark.columns
])

# Exibe o resultado em formato de tabela no Databricks
display(df_resumo_nulos)

Fui verificar esses casos nulos e são justamete os alunos que não possuem caderno ou presença igual a 1. Quando são zero, a proeficiencia fica nula. Provavelmente são alunos que tiveram alguma exceção na avaliação. Pontuando isso aqui pois será necessário utilzar essa info nas contruções das camadas silver e gold


In [0]:
#Tentei fazer a análise "na unha", mas aí perguntei pro GPT se havia uma forma de verificar as colunas agrupadas de forma massiva os 'nulls'
#Avaliador tenha piedade de mim.

# 1. Identifica automaticamente todas as colunas de notas (remove as colunas de agrupamento)
colunas_agrupamento = ["ano"]
colunas_notas = [c for c in df_spark.columns if c not in colunas_agrupamento]

# 2. Cria dinamicamente a lista de agregações para cada coluna
expressoes_agg = [
    F.sum(F.col(c).isNull().cast("int")).alias(f"nulos_{c}") 
    for c in colunas_notas
]

# 3. Executa o agrupamento passando a lista gerada (*expressoes_agg)
df_nulos = df_spark.groupBy(colunas_agrupamento).agg(*expressoes_agg)

# Exibe o resultado na tela
display(df_nulos)

Nenhuma das bases possui UF, tive que procurar o de-para na internet para poder realizar os cruzamentos.

Obtive o arquivo do site do IBGE disponibilizado neste link https://www.ibge.gov.br/explica/codigos-dos-municipios.php

Último acesso em 11/07/2026 às 17:54

In [0]:
%sql
select * from workspace.bronze.b_municipios_br